# Lecture 6: Text Splitters & Chunking Strategies (Chote Tukde Banane Ki Hikmat-e-Amli)

**Course:** NLP with LangChain | **Platform:** Hope to Skill  
**Duration:** ~20 minutes | **Level:** Intermediate  

---

## Bada Manzar (The Big Picture)

Lecture 5 mein hum ne sikha ke documents ko **load** kaise karte hain. Lekin masla yeh hai:

> 100 safho (pages) ki PDF ek boht badi text ki deewar ban jati hai.  
> Ek LLM usay ek sath process nahi kar sakta. Ek search engine us se jawab match nahi kar sakta.  
> Humne isay **smart, chote aur ba-maani tukdon mein katna** hota hai — isay hi **Chunking** kehte hain.

**Kisi bhi RAG system mein Chunking sab se aham faislon mein se ek hai.**  
Kharab chunks = kharab retrieval = kharab jawabaat, chahe aapka LLM kitna hi behtareen kyun na ho.

### Aap Kya Sikhenge (What You Will Learn)

| # | Topic | Haqiqi Dunya Ki Misal (Real-World Analogy) |
|---|-------|--------------------------------------------|
| 1 | Hum documents ki chunking kyun karte hain | Pizza serve karne se pehle uske slices kyun karte hain |
| 2 | Chunk size — Goldilocks ka masla | Slices boht chote ya boht bade hona |
| 3 | Overlap — chunks aapas mein text share kyun karte hain | Kitab ke safhey jo pichli line ko dohrate hain |
| 4 | RecursiveCharacterTextSplitter | Smart pizza cutter |
| 5 | Doosre splitters | Khas kaamon ke liye khas tools |
| 6 | Metadata ko mehfooz rakhna (Preservation) | Kabhi yeh na bhoolna ke chunk kahan se aaya tha |
| 7 | Hands-on: split, compare, choose | Khud bana kar dekhein! |

> **Aham Nuqta (Key Insight):** Chunking wo jagah hai jahan aksar RAG pipelines khamoshi se fail ho jati hain.  
> Is par aboor hasil kar lein, aur aap 90% beginners se aage nikal jayenge.

---

## 0. Environment Setup

Aj jin packages ki zaroorat padegi, unhe install karne ke liye is cell ko **ek baar** run karein.  
Agar aap ne Lecture 5 mein yeh pehle se install kar liye hain, toh aap tayyar hain!

In [ ]:
# Install required packages (run once, then you can skip this cell)
%pip install langchain langchain-community langchain-text-splitters pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## 1. Hum Documents Ki Chunking Kyun Karte Hain?

### Pizza Ki Misal (The Pizza Analogy)

Farz karein aap ne ek pizza order kiya. Chef aapko **poora uncut pizza** pakda deta hai.  
Aap usay aise nahi khate! Aapko uske slices karne padte hain.

Lekin aap usay **kaise kat te hain** yeh boht zaroori hai:
- **Boht Chote Slices** (boht chote murabba tukde) — har tukde par toppings naam barabar hongi, khane ka maza nahi aayega.
- **Boht Bade Slices** (adha pizza) — ek sath sambhalna boht mushkil ho jayega.
- **Bilkul Sahi Slices** (munasib slices) — har slice ek ba-maani, mukammal hissa hota hai.

Documents bhi bilkul isi tarah kaam karte hain. Chunking karne ki 3 bari wajohat yeh hain:

### Wajah 1: Context Window Ki Hudoos (Context Window Limits)
LLMs ek waqt mein ek khas hadd tak hi text padh sakte hain (jisay "context window" kehte hain).  
Aap LLM ko 500 safhey nahi bhej sakte — ya toh wo crash ho jayega ya zyada tar data ko ignore kar dega.

### Wajah 2: Retrieval Ki Aqsaat (Retrieval Precision)
Jab koi user sawal poochta hai, toh aap chahte hain ke **bilkul sahi 2-3 paragraphs** dhoond kar layein jo jawab dete hon — na ke 50 safhey utha kar de dein aur umeed karein ke LLM khud hi samajh jayega.

### Wajah 3: Maani Ka Rabt (Semantic Coherence)
Har chunk mein **ek mukammal khayal/idea** hona chahiye. Agar aap jumle ke beech mein se kat dein, toh wo chunk be-maani ho jata hai.

> **Khulasa:** Kharab chunking = kharab RAG. Aapka LLM chahe kitna hi acha kyun na ho, agar aap usay be-tukay chunks denge, toh jawabaat bhi be-tuka hi milenge.

In [1]:
# Aayein dekhte hain ke ek haqiqi misal ke sath chunking kyun zaroori hai
# Pehle, Lecture 5 ke data folder se apni sample article load karte hain

from langchain_community.document_loaders import TextLoader

txt_loader = TextLoader(

    file_path="data/nlp_article.txt",
    encoding="utf-8"

)

txt_docs = txt_loader.load()

# Yeh document kitna bada hai?
# documents[0] list mein se pehla (aur aik hi) document hasil karta hai
full_text = txt_docs[0].page_content

# len() string mein majood kul characters ki tadad count karta hai
print(f"Document length: {len(full_text):,} characters")

# Andazan tokens ka hisab: English mein 1 token ~ 4 characters ke barabar hota hai
# // integer division hai (jo divide karke decimal point khatam kar deta hai)
print(f"That's roughly {len(full_text) // 4:,} tokens")

# [:300] preview ke tor par sirf pehle 300 characters dikhata hai
# Poora document dekhne ke liye istemal karein: print(full_text)
print(f"\nFirst 300 characters:")
print(full_text[:300])

C:\Users\Muhammad Yahya\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document length: 8,039 characters
That's roughly 2,009 tokens

First 300 characters:
Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objectiv


#### Abhi Kya Hua?

Hum ne lagbhag 8,000 characters ka ek article load kiya. Yeh andazan 2,000 tokens bante hain.  
Yeh ziada tar LLMs ke liye kafi chota hai, lekin farz karein 200 safho (pages) ki PDF ho —  
toh usme 500,000+ characters ho sakte hain. Itna bara data ek sath bhenjna bilkul na-mumkin hai!

**Aasan Hisab (Quick math):** 1 token ~ 4 characters (English ke liye andazatan).  
GPT-4 ki context window ~128K tokens hai. Claude ki ~200K tokens hai.  
Lekin sirf is waja se ke poora document fit *aa sakta* hai, iska yeh matlab nahi ke yeh *karegar* bhi hoga — chote aur **focused chunks** hamesha behtareen retrieval results dete hain.

---

## 2. Chunk Size — Goldilocks Ka Masla (The Goldilocks Problem)

Sahi chunk size ka intekhab karna bilkul "Goldilocks aur teen bhaluon" ki kahani jaisa hai:  
na zyada chota, na zyada bara, balki bilkul munasib (just right).

### Alag Alag Sizes Par Kya Hota Hai?

| | Boht Chota (<200 chars) | Bilkul Sahi (500-1500 chars) | Boht Bara (>2000 chars) |
|---|---|---|---|
| **Misal ka chunk** | `"NLP is a branch of..."` | `"NLP is a branch of AI that helps computers understand language..."` | `"NLP is a branch of AI that... [3 pages of text] ...boht ziada noise!"` |
| **Masla** | Context khatam! Khayal samajh nahi aata. | Har chunk mein ek mukammal khayal/idea. | Search results mein ziada be-fizool text (noise). |
| **Haqiqi Misal** | Pizza ke boht chote tukde — topping milti hi nahi | Pizza ke munasib slices — ba-maani aur tasalli-bakhsh | Adha pizza ek hi plate mein — sambhalna mushkil |

### Munaasib Hissa (The Sweet Spot): 500-1500 Characters

| Chunk Size | Characters | Kiske Liye Achha Hai |
|-----------|------------|----------------------|
| Chota (Small) | 200-500 | FAQ ke jawabaat, chote paragraphs |
| Darmiyana (Medium) | 500-1000 | Aam istemal ke liye (behtareen default) |
| Bara (Large) | 1000-1500 | Technical documents, tafseeli wazahat |
| Boht Bara (Very Large) | 1500-2000 | Lambay articles, qanooni (legal) documents |

Iska koi ek "sahi" size nahi hai — yeh aap ke data aur istemal (use case) par munhasir karta hai.  
Hum is notebook mein aage chal kar alag alag sizes ke sath tajarba (experiment) karenge!

In [2]:
# Aayein dekhte hain ke HAQIQI text par ALAG ALAG chunk sizes kaise nazar aate hain
# LangChain istemal karne se PEHLE hum text ko khud slice karke samajhte hain

# [:2000] demo ke liye hamare article ke sirf pehle 2000 characters leta hai
# Poora article istemal karne ke liye is se replace karein: sample_text = full_text
sample_text = full_text[:2000]

# Hum 3 alag chunk sizes test karenge taake unka muazna (compare) kar saken
chunk_sizes = [100, 500, 1000]

# Yeh loop 3 baar chalega — list mein shamil har size ke liye ek baar
for size in chunk_sizes:
    # // integer division hai: 2000 // 500 = 4 chunks
    num_chunks = len(sample_text) // size

    # [:size] text ko shuru se le kar chunk size tak slice karta hai
    # Agar size=100, toh first_chunk = pehle 100 characters
    # Agar size=500, toh first_chunk = pehle 500 characters
    first_chunk = sample_text[:size]

    print(f"\n{'=' * 60}")
    print(f"CHUNK SIZE: {size} characters")
    print(f"2000 chars se banne wale chunks ki tadad: {num_chunks}")
    print(f"Pehle chunk ka preview:")
    print(f"  '{first_chunk}'")

    # # [size-20:size] pehle chunk ke AAKHRI 20 characters ko slice karta hai
    # # Yeh aapko dikhata hai ke chunk bilkul kahan par khatam/cut ho raha hai
    print(f"  --- jahan khatam hota hai: '...{sample_text[size - 20:size]}'")


CHUNK SIZE: 100 characters
2000 chars se banne wale chunks ki tadad: 20
Pehle chunk ka preview:
  'Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?
'
  --- jahan khatam hota hai: '...anguage Processing?
'

CHUNK SIZE: 500 characters
2000 chars se banne wale chunks ki tadad: 4
Pehle chunk ka preview:
  'Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.

NLP combines computational linguistics — rule-based modeling of human lan'
  --- jahan khatam hota hai: '...odeling of human lan'

CHUNK SIZE: 1000 characters
2000 chars se banne wale chunks ki tadad: 2
Pehle chunk ka

#### Abhi Kya Hua?

Ghaur karein ke har chunk **kahan par khatam (cut)** ho raha hai:

- **100 chars** — lagbhag jumle (sentence) ke beech mein se kat jata hai! Yeh chunk akele bilkul be-maani lagta hai.
- **500 chars** — ho sakta hai paragraph ke beech mein se kate, lekin kam az kam kuch hadd tak context samajh aata hai.
- **1000 chars** — isme lagbhag poora ek ya do paragraphs aa jate hain. Yeh ziada mufeed (useful) hai.

**Sada Slicing Ka Masla (The problem with simple slicing):** Yeh bilkul exact characters count par kat deta hai, chahe wo kisi lafz (word) ke beech mein hi kyun na ho! Isi liye humein smart splitters ki zaroorat hoti hai (jo aglay section mein aayenge).

---

## 3. Overlap — Chunks Aapas Mein Text Share Kyun Karte Hain?

### Overlap Ke Bina Masla (The Problem Without Overlap)

Farz karein yeh jumla do chunks ki sarhad (boundary) par maujood hai:

Chunk 1: "...BERT was released by Google."
Chunk 2: "It achieved state-of-the-art results on 11 NLP tasks."


Agar koi poochay *"What did BERT achieve?"*, toh dono mein se kisi ek chunk ke paas bhi poora jawab nahi hoga!  
Chunk 1 ko BERT ke baare mein pata hai lekin uski achievements ke baare mein nahi. Chunk 2 ke paas achievements toh hain lekin "It" se yeh pata nahi chalta ke kiski baat ho rahi hai.

### Hal: Overlap (The Solution: Overlap)

Overlap ka matlab hai ke **ek chunk ka aakhri hissa aglay chunk ke shuru mein dohraaya (repeat kiya) jata hai**:

Chunk 1: "...BERT was released by Google. It achieved state-of-the-art"
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Yeh hissa DONO chunks mein hai (overlap)
Chunk 2: "BERT was released by Google. It achieved state-of-the-art results on 11..."


Ab DONO chunks ke paas mukammal context maujood hai!

### Kitna Overlap Hona Chahiye? (How Much Overlap?)

**Aam Usool (Rule of thumb): Aap ke chunk size ka 10-20%**

| Chunk Size | Sifarish Karda Overlap | Kyun |
|-----------|--------------------|-----|
| 500 chars | 50-100 chars | ~1-2 jumlay context ke liye |
| 1000 chars | 100-200 chars | ~2-3 jumlay context ke liye |
| 1500 chars | 150-300 chars | ~3-5 jumlay context ke liye |

**Boht ziada overlap** (>30%) = storage zaya karta hai aur zaroorat se ziada information duplicate karta hai  
**Boht kam overlap** (0%) = boundaries par aham context zaya (lost) ho jata hai

In [5]:
# Aayein ek sadi misal ke sath samajhte hain ke overlap KAISE kaam karta hai
demo_text = (
    "BERT was released by Google in 2018. "
    "It achieved state-of-the-art results on 11 NLP tasks. "
    "GPT was developed by OpenAI. "
    "It uses a different training approach called autoregressive modeling."
)

chunk_size = 80
overlap = 30

print(f"Poora text ({len(demo_text)} chars): '{demo_text}'")
print(f"\nChunk size: {chunk_size} | Overlap: {overlap}")
print("=" * 60)

# step = agla chunk shuru karne se pehle hum kitne characters aage barhte hain
# Agar chunk_size=80 aur overlap=30 ho, toh step=50 hota hai
# Iska matlab hum 50 naye chars aage barhte hain, aur pichle chunk ke aakhri 30 chars dohraate hain
step = chunk_size - overlap
chunk_number = 1
start = 0


# Yeh while loop tab tak chunks banata rahega jab tak hum poora text cover na kar lein
# start hamari text mein mojooda position ko track karta hai
while start < len(demo_text):
    end = start + chunk_size

    # [start:end] text mein se ek chunk ko slice karta hai
    # e.g., demo_text[0:80] chars 0-79 deta hai, demo_text[50:130] chars 50-129 deta hai
    chunk = demo_text[start:end]

    # min(end, len(demo_text)) yeh yaqini banata hai ke hum text ki total length se aage ka index na dikhayein
    print(f"\nChunk {chunk_number} (chars {start}-{min(end, len(demo_text))}):")
    print(f"  '{chunk}'")

    chunk_number += 1
    start += step  # 50 step aage barhein, 80 nahi — isi se overlap banta hai

Poora text (189 chars): 'BERT was released by Google in 2018. It achieved state-of-the-art results on 11 NLP tasks. GPT was developed by OpenAI. It uses a different training approach called autoregressive modeling.'

Chunk size: 80 | Overlap: 30

Chunk 1 (chars 0-80):
  'BERT was released by Google in 2018. It achieved state-of-the-art results on 11 '

Chunk 2 (chars 50-130):
  'tate-of-the-art results on 11 NLP tasks. GPT was developed by OpenAI. It uses a '

Chunk 3 (chars 100-180):
  'eveloped by OpenAI. It uses a different training approach called autoregressive '

Chunk 4 (chars 150-189):
  'pproach called autoregressive modeling.'


#### Abhi Kya Hua?

- Har chunk 80 characters lamba hai, lekin hum agla chunk shuru karne se pehle sirf **50** characters aage barhte hain  
  (80 - 30 = 50)
- Iska matlab yeh hai ke har chunk ke aakhri 30 characters aglay chunk ke shuru mein dobara aate hain — isi ko **overlap** kehte hain
- Chunks par ghaur karein: aapko lagataar aane wale chunks ke beech dohraaya gaya (repeated) text nazar aayega

**`step = chunk_size - overlap` kyun?** Agar chunks 80 chars ke hon aur overlap 30 ho, toh hum window ko sirf 50 chars aage slide karte hain. Baqi bache 30 chars aglay chunk ke sath share ho jate hain. LangChain ke splitters andaruni tor par bilkul isi tarah kaam karte hain!